# Sprint 4 — BF (2.2.4 Calibración RMSC04 + 2.2.3 Recompensa Ejecutores) — Colab

Receta de instalación **corregida y probada el 30 sept 2026** (Python 3.9.23 + ABIDES `jpmorganchase/abides-jpmc-public` main).
Reemplaza a `tesis_prueba_uno.ipynb`, que fallaba en la Celda 3 (gym 0.18.0) y en la 5 (repo no clonado).

**Orden:** Celda 1 → (el runtime se reinicia solo) → Celdas 2 a 8 una vez por sesión → celdas de trabajo (9+).
Todo lo que usa ABIDES se corre con `conda run --no-capture-output -n abides ...` (el kernel del notebook NO es 3.9).

In [ ]:
# Celda 1 — condacolab (reinicia el runtime; es normal que "se caiga" la ejecución aquí)
# Se fija <0.2 porque 0.2.x cambia el método de instalación (Pixi) y puede romper la receta.
!pip install -q "condacolab<0.2"
import condacolab
condacolab.install()

In [ ]:
# Celda 2 — entorno conda 'abides' con Python 3.9 (idempotente)
import os
if not os.path.exists('/usr/local/envs/abides'):
    !conda create -n abides python=3.9 -y -q
else:
    print('Entorno abides ya existe')
!conda run -n abides python --version

In [ ]:
# Celda 3 — dependencias en 4 pasos (CORREGIDO). Por qué no un solo pip install:
#  - el env nuevo trae pip 25.x: el pin pip==23.3.2 no rige hasta terminar el comando,
#    y pip/setuptools modernos rechazan el requisito 'opencv-python>=3.' de gym 0.18.0
#  - gym 0.18.0 pide Pillow<=7.2.0, sin wheel para Python 3.9 -> --no-deps + Pillow<10
#  - ray 1.7.0 no importa con protobuf>=4 -> protobuf==3.20.3
# Es ESPERADO el aviso "gym 0.18.0 requires Pillow<=7.2.0, but you have pillow 9.5.0" (inofensivo: Pillow solo se usa para render).
%env PIP_DISABLE_PIP_VERSION_CHECK=1
!conda run -n abides python -m pip install -q pip==23.3.2 setuptools==57.5.0 wheel==0.38.4
!conda run -n abides python -m pip install -q "cython<3" numpy==1.22.0
!conda run -n abides python -m pip install -q --no-build-isolation --no-deps gym==0.18.0
!conda run -n abides python -m pip install -q --no-build-isolation \
    numpy==1.22.0 "cloudpickle>=1.2,<1.7" "pyglet>=1.4,<=1.5.0" "Pillow<10" \
    pandas==1.2.4 scipy==1.10.0 coloredlogs==15.0.1 psutil==5.8.0 tqdm==4.61.1 \
    pomegranate==0.14.5 "ray[tune]==1.7.0" "protobuf==3.20.3" \
    pyarrow==11.0.0 matplotlib==3.5.3 pytest
!conda run -n abides python -m pip install -q "gymnasium==0.29.1" "importlib-metadata>=4.8" farama-notifications --no-deps

In [ ]:
# Celda 4 — ABIDES (core, markets, gym) desde el repo oficial, dentro del env (idempotente)
import os
if not os.path.exists('/content/abides-jpmc-public'):
    !git clone -q https://github.com/jpmorganchase/abides-jpmc-public.git /content/abides-jpmc-public
for pkg in ['abides-core', 'abides-markets', 'abides-gym']:
    !cd /content/abides-jpmc-public/{pkg} && conda run -n abides python setup.py -q install > /content/install_{pkg}.log 2>&1 && echo "{pkg} OK" || (echo "{pkg} FALLO, ver /content/install_{pkg}.log"; tail -20 /content/install_{pkg}.log)

In [ ]:
# Celda 5 — verificación de versiones (esperado: gym 0.18.0 | numpy 1.22.0 | pandas 1.2.4 | gymnasium 0.29.1 | ray 1.7.0)
!conda run -n abides python -c "import gym, numpy, pandas, gymnasium, ray, abides_core, abides_markets, abides_gym; print(gym.__version__, numpy.__version__, pandas.__version__, gymnasium.__version__, ray.__version__)"

In [ ]:
# Celda 6 — repo slippage. Cambiar BRANCH a la rama BF cuando Claude Code la haya pusheado.
BRANCH = 'main'   # luego: 'feature/2.2.4-2.2.3-rmsc04-recompensa-BF'
import os
if not os.path.exists('/content/slippage'):
    !git clone -q https://github.com/Vatelfil/slippage.git /content/slippage
!cd /content/slippage && git fetch -q origin && git checkout -q {BRANCH} && git pull -q && git log --oneline -1

In [ ]:
# Celda 7 — datos (los .parquet NO están en git). Subir una vez a Drive los zips de backups/
# (snapshot_2026-08-23.zip y snapshot_2026-09-27.zip) en la carpeta DRIVE_DIR.
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/slippage_colab'   # ajustar si usas otra carpeta
RESULTS_DIR = f'{DRIVE_DIR}/resultados'
import os; os.makedirs(RESULTS_DIR, exist_ok=True)
for snap in ['2026-08-23', '2026-09-27']:
    z = f'{DRIVE_DIR}/snapshot_{snap}.zip'
    if os.path.exists(z):
        !unzip -q -o "{z}" -d /content/slippage && echo "snapshot {snap} OK"
    else:
        print(f'FALTA {z}')
!ls /content/slippage/data/processed/

In [ ]:
# Celda 8 — smoke test ABIDES (debe terminar con "ABIDES-Gym es funcional")
!cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python src/envs/test_abides.py 2>&1 | tail -5

## Experimento rápido H5/H6 — unidades de `fund_vol` y unidad de cuenta (no depende de código nuevo)
Reproduce la tabla del plan con tus datos: compara el `fund_vol` heurístico actual de la plantilla contra la conversión corregida `σ_5min · r_bar / √(300e9)`, en centavos y en "1 unidad = 1 CLP".

In [ ]:
%%writefile /content/exp_fundvol.py
import json, sys, time
import numpy as np, pandas as pd
from abides_core import abides
from abides_markets.configs import rmsc04

SNAP = sys.argv[1] if len(sys.argv) > 1 else "2026-08-23"
TICKER, TRAMO = "FALABELLA", "apertura"
px = pd.read_parquet(f"/content/slippage/data/processed/clean_5m_{SNAP}/_combined.parquet")
p_med = float(px.loc[px["ticker"] == TICKER, "close_raw"].median())
sig5 = json.load(open("/content/slippage/data/calibration/poisson_params_2026-08-23.json"))["params"][TICKER][TRAMO]["obs_return_std"]
NS_5MIN = 300e9

casos = {
    "default_rmsc04":        (int(p_med * 100), 5e-5),
    "heuristica_plantilla":  (int(p_med * 100), sig5 * p_med * 100),
    "corregida_centavos":    (int(p_med * 100), sig5 * p_med * 100 / np.sqrt(NS_5MIN)),
    "corregida_1unidad_CLP": (int(round(p_med)), sig5 * p_med / np.sqrt(NS_5MIN)),
}
print(f"{TICKER} {TRAMO} snapshot {SNAP}: P mediano={p_med:.2f} CLP  sigma_5min real={sig5*1e4:.1f} bps")
for nombre, (r_bar, fv) in casos.items():
    t = time.time()
    cfg = rmsc04.build_config(seed=1, end_time="10:30:00", r_bar=r_bar, fund_vol=fv,
                              stdout_log_level="ERROR", log_orders=False)
    end = abides.run(cfg)
    L1 = end["agents"][0].order_books["ABM"].get_L1_snapshots()
    bb = pd.DataFrame(L1["best_bids"], columns=["t", "p", "q"]).set_index("t")["p"]
    ba = pd.DataFrame(L1["best_asks"], columns=["t", "p", "q"]).set_index("t")["p"]
    df = pd.concat([bb, ba], axis=1, keys=["b", "a"]).ffill().dropna()
    df.index = pd.to_datetime(df.index)
    mid = ((df.a + df.b) / 2).resample("1min").last().dropna()
    r = np.log(mid).diff().dropna()
    spr = ((df.a - df.b) / ((df.a + df.b) / 2) * 1e4).median()
    print(f"  {nombre:24s} fund_vol={fv:10.3g}  vol5m={r.std()*np.sqrt(5)*1e4:9.1f} bps  "
          f"spread_med={spr:6.2f} bps  mid_ini/fin={mid.iloc[0]:.0f}/{mid.iloc[-1]:.0f}  ({time.time()-t:.0f}s)")

In [ ]:
!cd /content && conda run --no-capture-output -n abides python /content/exp_fundvol.py 2026-08-23 2>&1 | grep -v UserWarning

## Celdas de trabajo (requieren los scripts que crea Claude Code en la rama BF)
Antes: en la Celda 6 poner `BRANCH` = rama BF y re-ejecutarla. Los resultados se escriben en `RESULTS_DIR` (Drive) para no perderlos si Colab se desconecta; los scripts son reanudables.

In [ ]:
# Celda 9 — búsqueda en grilla (bloque B de 2.2.4)  ~1 h
!cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python scripts/colab/rmsc04_grid_search.py \
    --ticker FALABELLA --snapshot 2026-08-23 --seeds 3 --out-dir "{RESULTS_DIR}" 2>&1 | grep -v UserWarning

In [ ]:
# Celda 10 — validación final (KS por tramo con >=10 semillas, momentos vs objetivos_validacion)
!cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python scripts/colab/rmsc04_validate.py \
    --ticker FALABELLA --snapshot 2026-08-23 --seeds 10 --grid-json "{RESULTS_DIR}/rmsc04_grid_FALABELLA_2026-08-23.json" \
    --out-dir "{RESULTS_DIR}" 2>&1 | grep -v UserWarning

In [ ]:
# Celda 11 — antes/después de 2.2.5: collect_abides_stats con la config calibrada (para Paolo)
!cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python src/envs/collect_abides_stats.py \
    --calibrated-json "{RESULTS_DIR}/rmsc04_ipsa_FALABELLA_2026-08-23.json" --out-dir "{RESULTS_DIR}" 2>&1 | grep -v UserWarning

In [ ]:
# Celda 12 — barrido de beta en ABIDES calibrado (2.2.3, fase 1)
!cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python scripts/colab/beta_sweep_abides.py \
    --calibrated-json "{RESULTS_DIR}/rmsc04_ipsa_FALABELLA_2026-08-23.json" --out-dir "{RESULTS_DIR}" 2>&1 | grep -v UserWarning

In [ ]:
# Celda 13 — empaquetar resultados y descargarlos (después copiarlos al repo local)
!cd "{RESULTS_DIR}" && zip -q -r /content/resultados_sprint4_BF.zip . && ls -la
from google.colab import files
files.download('/content/resultados_sprint4_BF.zip')